In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

data_path = "/kaggle/input/competitions/playground-series-s6e2/"

train = pd.read_csv(data_path + "train.csv")
test = pd.read_csv(data_path + "test.csv")

print("Train Shape is: ", train.shape)
print("Train Shape is: ", test.shape)

train.head()

# EDA

In [ ]:
#Used for Exploratory Data Analysis (EDA) to check for class imbalance
train['Heart Disease'].value_counts(normalize = True)

In [ ]:
#visually check for class imbalance by plotting the frequency distribution of your target variable
sns.countplot(x='Heart Disease', data = train)
plt.title("Target Distribution")
plt.show()

In [ ]:
#Missing values check
train.isnull().sum()

In [ ]:
# Feature Overview
train.describe()

# Baseline CatBoost model with CV

In [ ]:
#Step 1: Separate features & target
TARGET = "Heart Disease"

X = train.drop(columns=[TARGET, "id"])
y = train[TARGET]

X_test = test.drop(columns=["id"])


In [ ]:
#Step 2: Identify categorical features
#CatBoost handles categoricals natively
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
categorical_features


In [ ]:
#Step 3: Install & import CatBoost
!pip install catboost

In [ ]:
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

In [ ]:
#Step 4: Cross-validation training
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"Fold {fold+1}")
    
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=100
    )
    
    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        cat_features=categorical_features,
        use_best_model=True
    )
    
    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
    test_preds += model.predict_proba(X_test)[:, 1] / N_SPLITS

In [ ]:
#Step 5: CV score
cv_auc = roc_auc_score(y, oof_preds)
cv_auc

In [ ]:
#Phase 3 — submission file
submission = pd.DataFrame({
    "id": test["id"],
    "Heart Disease": test_preds
})

submission.to_csv("submission.csv", index=False)
submission.head()


# MODEL

In [ ]:
#Step 1 — Train final CatBoost on full data
from catboost import CatBoostClassifier

final_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100
)

final_model.fit(
    X, y,
    cat_features=categorical_features
)


In [ ]:
#Save model
MODEL_PATH = "catboost_heart_disease_model.cbm"
final_model.save_model(MODEL_PATH)

MODEL_PATH

In [ ]:
#Save feature list
import json

feature_info = {
    "features": X.columns.tolist(),
    "categorical_features": categorical_features,
    "target": "Heart Disease",
    "metric": "ROC-AUC"
}

with open("model_features.json", "w") as f:
    json.dump(feature_info, f, indent=4)
